# Date2Date Seq2Seq：日期格式转换实验

目标：训练一个字符级 seq2seq 模型，将日期字符串从：

`YYYY-MM-DD`

转换为：

`DD/MM/YYYY`

例如：

`2026-06-28 -> 28/06/2026`

本 notebook 当前阶段目标：

1. 搭建最小可运行的 Encoder-Decoder RNN；
2. 跑通训练、生成、评估闭环；
3. 记录 baseline 的失败现象；
4. 分析为什么 vanilla RNN 当前效果差；
5. 为下一轮改进做 checkpoint。

## 1. Imports、随机种子、全局配置

In [20]:
import torch
import torch.nn as nn
import random
from torch import Tensor
from typing import Optional
from torch.nn.modules.loss import CrossEntropyLoss


In [21]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cpu


## 2. 字符表与张量转换工具


In [22]:
# 本实验室字符级建模，词表中每个字符都是一个token。
chars = "0123456789-/"
# - `<SOS>`：decoder 生成时的起始 token；
# - `<EOS>`：target 序列结束标记。
special_tokens = ["<SOS>", "<EOS>"]

itos = special_tokens + list(chars)
stoi = {ch: i for i, ch in enumerate(itos)}


SOS_token = stoi["<SOS>"]
EOS_token = stoi["<EOS>"]

vocab_size = len(itos)

print(itos)
print(stoi)

['<SOS>', '<EOS>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '/']
{'<SOS>': 0, '<EOS>': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, '-': 12, '/': 13}


In [23]:
def tensor_from_string(s, add_eos=False):
    ids = [stoi[ch] for ch in s]
    if add_eos:
        ids.append(EOS_token)

    return torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

def string_from_tensor(tensor):
    chars = []
    tensor = tensor.squeeze(0)
    for token in tensor:
        idx = token.item()
        if idx == EOS_token:
            break
        if idx == SOS_token:
            continue
        chars.append(itos[idx]) 
    return "".join(chars)

In [24]:

x = tensor_from_string("2002-01-23")
print(x, x.shape)
y = tensor_from_string("23/01/2002", add_eos=True)
print(y, y.shape)


tensor([[ 4,  2,  2,  4, 12,  2,  3, 12,  4,  5]]) torch.Size([1, 10])
tensor([[ 4,  5, 13,  2,  3, 13,  4,  2,  2,  4,  1]]) torch.Size([1, 11])


In [25]:
print(string_from_tensor(x))
print(string_from_tensor(y))

2002-01-23
23/01/2002


3. 数据生成与样本检查

In [26]:
# 生成的样本符合
# 输入长度固定为 10：YYYY-MM-DD。日和月都是2位，且日以28天计数
# 输出长度固定为 10：DD/MM/YYYY
# 再加 EOS 后 target 长度为 11。
def make_sample(TRAIN_SIZE):
    ep = TRAIN_SIZE
    inputs = []
    targets = []
    for i in range(ep):
        year = random.randint(1950, 2050)
        day = random.randint(1, 28)
        month = random.randint(1, 12)
        input = f"{year}-{month:02d}-{day:02d}"
        inputs.append(input)
        target = f"{day:02d}/{month:02d}/{year}"
        targets.append(target)
    return inputs, targets

inputs, targets = make_sample(100)
print(inputs[0:10])
print(targets[0:10])

print("len(inputs)== len(targets)",len(inputs)== len(targets))

['2031-01-04', '2044-04-09', '1978-12-05', '1963-12-22', '2019-10-03', '2004-01-02', '1961-04-07', '2014-01-20', '2021-12-07', '2033-09-23']
['04/01/2031', '09/04/2044', '05/12/1978', '22/12/1963', '03/10/2019', '02/01/2004', '07/04/1961', '20/01/2014', '07/12/2021', '23/09/2033']
len(inputs)== len(targets) True


In [27]:
fixed_samples = [
    ("2002-01-23", "23/01/2002"),
    ("1999-12-08", "08/12/1999"),
    ("2020-03-07", "07/03/2020"),
    ("1950-11-28", "28/11/1950"),
    ("2048-06-15", "15/06/2048"),
]

## 4. Encoder / Decoder 模型定义

本实验先使用最基础的 Encoder-Decoder RNN，不加 attention。

Encoder 将整个输入日期编码成最后一个 hidden state。
Decoder 使用这个 hidden state 作为初始状态，逐字符生成目标日期。

这个 baseline 的限制是：整个输入序列的信息都被压缩到一个 hidden vector 中，
对于日期格式转换这种位置拷贝任务，可能不够稳定。

In [28]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.0)

    def forward(self, input):
        # 要限定 input的shape
        # input.shape (batch_size, seqlen)
        input = self.embedding(input)
        input = self.dropout(input)
        # hidden不受 batch_first=True的控制，其shape仍然为(direction*num,batch_size,hidden_size)
        # output.shape 是 (batch_size,seq_len,direction*hidden_size)
        output, hidden = self.rnn(input)
        return output, hidden


In [29]:


class DecoderRNN(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, output_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_hidden: Tensor, target_tensor: Optional[Tensor] = None, max_len: int = 20):
        # encoder_output是encoder输出的 hiddeng向量
        # target_tensor.shape is (batch_size, seq_len)
        # foward.shape is (batch_size, seq_len,vocab_size)
        batch_size = encoder_hidden.shape[1]
        hidden = encoder_hidden
        outputs = []
        # input_token的格式是 (batch_size,seq_len)
        input_token = torch.full((batch_size, 1), SOS_token, dtype=torch.long) #设置起始词元
        loop_len = max_len
        if target_tensor is not None:
            loop_len = target_tensor.shape[1]

        for i in range(loop_len):
            logits, hidden, pred_token = self.forward_step(hidden, input_token) 
            outputs.append(logits)

            if target_tensor is not None:
                input_token = target_tensor[:,i].unsqueeze(1)
            else :
                input_token = pred_token.detach()
        decoder_outputs = torch.stack(tensors=outputs, dim=1)
        return decoder_outputs, hidden
    
    def forward_step(self, hidden, input_token):
        # input_token.shape是（batch_size,seq_len)
        embedded = self.embedding(input_token)
        # rnn的hidden输入是(batch_size,seq_len,hidden)
        output, hidden = self.rnn(embedded, hidden)
        # hidden不受 batch_first=True的控制，其shape仍然为(direction*num,batch_size,hidden_size)
        # output.shape 是 (batch_size,seq_len,direction*hidden_size)
        logits = self.out(hidden[-1]) # hidden[-1]表示最后一个num
        pred_token = logits.argmax(dim = -1,keepdim= True)
        return logits, hidden, pred_token

    

In [30]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)

x = tensor_from_string("2002-1-23")
print("x", x.shape)

encoder_output, encoder_hidden = encoder.forward(x);
print("encoder_output:", encoder_output.shape)
print("encoder_hidden:", encoder_hidden.shape)

x torch.Size([1, 9])
encoder_output: torch.Size([1, 9, 5])
encoder_hidden: torch.Size([1, 1, 5])


In [31]:
input_token = torch.full((1, 1), SOS_token, dtype=torch.long)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)
logits, decoder_hidden, pred_token = decoder.forward_step(encoder_hidden, input_token)

print("decoder_hidden:", decoder_hidden.shape)
print("logits:", logits.shape)
print("pred_token", pred_token)


decoder_hidden: torch.Size([1, 1, 5])
logits: torch.Size([1, 14])
pred_token tensor([[8]])


In [32]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)

x = tensor_from_string("2002-01-23")
y = tensor_from_string("23/01/2002", add_eos=True)

encoder_output, encoder_hidden = encoder(x)
decoder_outputs, decoder_hidden = decoder(encoder_hidden, y)

print("decoder_outputs:", decoder_outputs.shape)
print("decoder_hidden:", decoder_hidden.shape)
print("target:", y.shape)


decoder_outputs: torch.Size([1, 11, 14])
decoder_hidden: torch.Size([1, 1, 5])
target: torch.Size([1, 11])


## 5. 训练、生成、评估函数

5.1 train_one_sample

In [33]:

def train_one_sample(input_str, target_str, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    input_tensor = tensor_from_string(input_str) 
    target_tensor = tensor_from_string(target_str, True)
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    encoder_output, encoder_hidden = encoder(input_tensor)
    decoder_outputs, decoder_hidden = decoder(encoder_hidden, target_tensor)
    logits = decoder_outputs.reshape(-1, vocab_size)
    target = target_tensor.reshape(-1)
    loss = criterion(logits, target)
    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item()


5.2 generate

In [34]:
def generate(input_str, encoder, decoder, max_len=20):
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        input_tensor = tensor_from_string(input_str)
        encoder_output, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden = decoder(encoder_hidden, None, max_len)
        # decoder_outputs.shape (batch,seq_len,vocab_size)
        pred_tokens = decoder_outputs.argmax(dim=2)
        outputs = []
        for token in pred_tokens[0]:
            if token.item() == EOS_token:
                break
            outputs.append(token)
        if len(outputs) == 0:
            return ""
        
    
    return string_from_tensor(torch.stack(outputs).unsqueeze(0))

5.3 eval_samples

In [35]:


def eval_samples(samples, encoder, decoder, verbose=False):
    encoder.eval()
    decoder.eval()

    correct_count = 0

    for input_str, target_str in samples:
        pred = generate(input_str, encoder, decoder)
        correct = pred == target_str

        if correct:
            correct_count += 1

        if verbose:
            print("input:  ", input_str)
            print("pred:   ", pred)
            print("target: ", target_str)
            print("correct:", correct)
            print("---")

    total = len(samples)
    acc = correct_count / total

    return correct_count, total, acc


In [36]:
def char_accuracy(samples, encoder, decoder):
    total_chars = 0
    correct_chars = 0

    for input_str, target_str in samples:
        pred = generate(input_str, encoder, decoder, max_len=20)
        max_compare_len = min(len(pred), len(target_str))

        for i in range(max_compare_len):
            total_chars += 1
            if pred[i] == target_str[i]:
                correct_chars += 1

        total_chars += max(0, len(target_str) - max_compare_len)

    return correct_chars / total_chars

## 6. Baseline 实验：当前 vanilla RNN seq2seq

In [37]:
# 循环训练
EPOCHS = 10
TEST_SIZE = 100
TRAIN_SIZE = 1000
HIDDEN_SIZE = 256
LR = 0.001

loss_group = TRAIN_SIZE//10
epoch_group = EPOCHS//10

inputs, targets = make_sample(TRAIN_SIZE)
samples = list(zip(inputs, targets))
# samples = fixed_samples

encoder = EncoderRNN(vocab_size, HIDDEN_SIZE)
decoder = DecoderRNN(vocab_size, HIDDEN_SIZE,vocab_size)

encoder_optimizer = torch.optim.Adam(encoder.parameters(), LR)
decoder_optimizer = torch.optim.Adam(decoder.parameters(), LR)

criterion = nn.CrossEntropyLoss()
for j in range(EPOCHS):
    encoder.train()
    decoder.train()
    epoch_loss = 0

    random.shuffle(samples)
    for i, (input_str, target_str) in enumerate(samples):
        loss = train_one_sample(input_str, target_str, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        epoch_loss += loss
    
    if (j + 1) % epoch_group == 0:
        print("EPOCHS:", j + 1, ", avg_epoch_loss:", epoch_loss / len(samples))
        eval_samples(fixed_samples, encoder, decoder, verbose=True)
        

    
            
train_correct, train_total, train_acc = eval_samples(samples, encoder, decoder )
print("train accuracy:", train_correct, "/", len(samples), train_acc)

train_char_accuracy = char_accuracy(samples, encoder, decoder )
print("train char_accuracy:", train_char_accuracy)

test_inputs, test_targets = make_sample(TEST_SIZE)
test_samples = list(zip(test_inputs, test_targets))
test_correct, test_total, test_acc = eval_samples( test_samples, encoder, decoder)
print("test accuracy:", test_correct, "/", len(test_samples), test_acc)
test_char_accuracy = char_accuracy(test_samples, encoder, decoder )
print("test char_accuracy:", test_char_accuracy)

EPOCHS: 1 , avg_epoch_loss: 0.6793907687664033
input:   2002-01-23
pred:    23/01/1997
target:  23/01/2002
correct: False
---
input:   1999-12-08
pred:    08/12/1993
target:  08/12/1999
correct: False
---
input:   2020-03-07
pred:    07/03/2006
target:  07/03/2020
correct: False
---
input:   1950-11-28
pred:    28/11/1999
target:  28/11/1950
correct: False
---
input:   2048-06-15
pred:    15/06/2006
target:  15/06/2048
correct: False
---
EPOCHS: 2 , avg_epoch_loss: 0.32943231381103394
input:   2002-01-23
pred:    23/02/2012
target:  23/01/2002
correct: False
---
input:   1999-12-08
pred:    08/12/1997
target:  08/12/1999
correct: False
---
input:   2020-03-07
pred:    07/03/2023
target:  07/03/2020
correct: False
---
input:   1950-11-28
pred:    28/11/1955
target:  28/11/1950
correct: False
---
input:   2048-06-15
pred:    15/06/2034
target:  15/06/2048
correct: False
---
EPOCHS: 3 , avg_epoch_loss: 0.19800057299342005
input:   2002-01-23
pred:    23/01/2020
target:  23/01/2002
correct

EPOCHS = 10
TEST_SIZE = 100
TRAIN_SIZE = 2000
HIDDEN_SIZE = 256
LR = 0.001

loss_group = TRAIN_SIZE//10
epoch_group = EPOCHS//10

EPOCHS: 10 , avg_epoch_loss: 0.5933430655971169
input:   2002-01-23
pred:    23/02/1964
target:  23/01/2002
correct: False
---
input:   1999-12-08
pred:    08/12/1964
target:  08/12/1999
correct: False
---
input:   2020-03-07
pred:    07/05/1964
target:  07/03/2020
correct: False
---
input:   1950-11-28
pred:    28/11/2022
target:  28/11/1950
correct: False
---
input:   2048-06-15
pred:    15/03/1964
target:  15/06/2048
correct: False
---
train accuracy: 11 / 2000 0.0055
train char_accuracy: 0.65955
test accuracy: 0 / 100 0.0
test char_accuracy: 0.676

dd/mm比较正确，但是yyyy准确率低。准备换一下模型试一下。

In [ ]:
更换成gru
EPOCHS: 10 , avg_epoch_loss: 0.033306556500232544
input:   2002-01-23
pred:    23/01/2002
target:  23/01/2002
correct: True
---
input:   1999-12-08
pred:    08/12/1997
target:  08/12/1999
correct: False
---
input:   2020-03-07
pred:    07/03/2020
target:  07/03/2020
correct: True
---
input:   1950-11-28
pred:    28/11/1950
target:  28/11/1950
correct: True
---
input:   2048-06-15
pred:    15/06/2048
target:  15/06/2048
correct: True
---
train accuracy: 923 / 1000 0.923
train char_accuracy: 0.9902
test accuracy: 93 / 100 0.93
test char_accuracy: 0.99


7. Attention

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size: int):
        super().__init__()
        self.W_s = nn.Linear(hidden_size, hidden_size)  # 作用在 decoder hidden 上
        self.W_h = nn.Linear(hidden_size, hidden_size)  # 作用在 encoder_outputs 上
        self.v_a = nn.Linear(hidden_size, 1)    

    def forward(self, decoder_hidden_last: Tensor, encoder_outputs: Tensor):
        """
        decoder_hidden_last: (batch_size, hidden_size)        # decoder 上一步的 hidden，即 hidden[-1]
        encoder_outputs:     (batch_size, src_len, hidden_size)
        返回:
            context:      (batch_size, 1, hidden_size)
            attn_weights: (batch_size, src_len)
        """
        e_ts = []
        for i in range(encoder_outputs.shape[1]):
            e_t = self.v_a(torch.tanh(self.W_s(decoder_hidden_last) + self.W_h(encoder_outputs[:, i, :])))
            e_ts.append(e_t)
        e_ts = torch.stack(tensors=e_ts, dim=1)
        attn_weights = torch.softmax(e_ts, dim=1)
        context = encoder_outputs * attn_weights
        return context, attn_weights        # 把对齐向量压成一个分数